# Model 2 — Parallel CNN

Song song: các nhánh tích chập 1x1, tích chập 3x3 và pooling trích xuất đặc trưng rồi ghép bằng Concatenate.

## 1. Cấu hình và thư viện

Khối này nạp TensorFlow, NumPy và các hàm đánh giá; đặt seed để kết quả có tính lặp lại tốt hơn; chọn `cut` hoặc `raw`; tìm thư mục gốc và chuẩn bị nơi lưu kết quả. Hãy dùng cùng một dataset và seed khi đối chiếu các kiến trúc.

In [ ]:
import csv
import json
import os
import time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models

DATASET = "raw"
MODEL_NAME = "parallel_cnn"
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Find the project root even when this notebook is opened from a nested folder.
ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "data" / DATASET).is_dir():
        ROOT = candidate
        break

data_dir = ROOT / "data" / DATASET
if not data_dir.is_dir():
    raise FileNotFoundError(f"Cannot find dataset folder: {data_dir}")

OUTPUT_DIR = ROOT / "src" / "Model_2" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Dataset: {data_dir}")

KeyboardInterrupt: 

## 2. Xây dựng kiến trúc CNN

Song song: các nhánh tích chập 1x1, tích chập 3x3 và pooling trích xuất đặc trưng rồi ghép bằng Concatenate.

In [ ]:
def build_model(number_of_classes):
    inputs = layers.Input(shape=(*IMAGE_SIZE, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(15 / 360)(x)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.RandomZoom((-0.1, 0.1), width_factor=(-0.1, 0.1))(x)
    x = layers.RandomBrightness(0.2)(x)
    x = layers.RandomContrast(0.2)(x)
    x = layers.RandomSaturation(0.2)(x)
    x = layers.Rescaling(1.0 / 255)(x)

    def conv_bn_relu(values, filters, kernel_size):
        values = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)(values)
        values = layers.BatchNormalization()(values)
        return layers.Activation("relu")(values)

    x = conv_bn_relu(x, 32, 3)
    x = layers.MaxPooling2D((2, 2))(x)

    # Three small branches learn features at different scales.
    branch_1 = conv_bn_relu(x, 32, 1)
    branch_2 = conv_bn_relu(x, 32, 3)
    branch_3 = layers.MaxPooling2D((3, 3), strides=1, padding="same")(x)
    branch_3 = conv_bn_relu(branch_3, 32, 1)
    x = layers.Concatenate()([branch_1, branch_2, branch_3])

    x = layers.MaxPooling2D((2, 2))(x)
    x = conv_bn_relu(x, 128, 3)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    x = conv_bn_relu(x, 256, 3)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(number_of_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name=MODEL_NAME)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## 3. Nạp và chuẩn bị dữ liệu

Ảnh được đọc theo cấu trúc `train/val/test/<lớp>`, resize đồng nhất và mã hóa nhãn one-hot cho `categorical_crossentropy`. Chuẩn hóa bằng `Rescaling(1/255)` đưa pixel về [0, 1].

Theo các phép augmentation của Model 1, chỉ trên dữ liệu train: lật ngang, xoay tối đa 15°, dịch chuyển tối đa 10% theo mỗi chiều, zoom khoảng 0.9–1.1, và biến đổi brightness/contrast/saturation với factor 0.2. Các lớp augmentation của Keras chỉ biến đổi ảnh khi `training=True`; validation và test không bị augmentation. Nếu hướng ảnh OCT có ý nghĩa lâm sàng thì cần cân nhắc bỏ lật ngang.

Validation dùng để theo dõi quá trình học; test được giữ riêng để đánh giá cuối.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 15

split_dirs = {split: data_dir / split for split in ("train", "val", "test")}
missing_dirs = [str(path) for path in split_dirs.values() if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError(f"Missing dataset split folders: {missing_dirs}")

train_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["train"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["val"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["test"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

class_names = train_ds.class_names
if val_ds.class_names != class_names or test_ds.class_names != class_names:
    raise ValueError("Train, validation and test class folders must match.")

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

print("Classes:", class_names)

## 4. Khởi tạo và kiểm tra mô hình

Tạo đầu ra theo số lớp lấy từ thư mục dữ liệu, sau đó kiểm tra kích thước batch ảnh, nhãn và đầu ra. Với bài toán này, đầu ra cần có 4 xác suất lớp; số tham số được in ra để hỗ trợ so sánh chi phí mô hình.

In [ ]:
model = build_model(number_of_classes=len(class_names))
images, labels = next(iter(train_ds))
print("Input batch:", images.shape)
print("Label batch:", labels.shape)
print("Model output:", model(images, training=False).shape)
print(f"Trainable parameters: {model.count_params():,}")

## 5. Huấn luyện và lưu checkpoint tốt nhất

Mô hình cập nhật trọng số từ tập train và theo dõi `val_loss` trên validation. `ModelCheckpoint` giữ lại checkpoint có validation loss tốt nhất; `EarlyStopping` dừng khi không còn cải thiện để hạn chế học quá mức. Test không tham gia chọn checkpoint.

In [ ]:
model_path = OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}.keras"

train_started = time.perf_counter()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    shuffle=False,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            str(model_path),
            monitor="val_loss",
            save_best_only=True,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
        ),
    ],
)

train_time = time.perf_counter() - train_started
print("Best checkpoint:", model_path)

## 6. Đánh giá trên tập test

Sau khi chọn checkpoint bằng validation, đánh giá một lần trên test. Accuracy cho biết tỷ lệ dự đoán đúng tổng thể; precision, recall và F1-score theo từng lớp cùng confusion matrix giúp nhìn rõ các lớp bị nhầm.

In [ ]:
best_model = tf.keras.models.load_model(model_path)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)
y_true = np.concatenate([np.argmax(labels.numpy(), axis=1) for _, labels in test_ds])
y_pred = np.argmax(best_model.predict(test_ds, verbose=0), axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0,
    output_dict=True,
)
matrix = confusion_matrix(y_true, y_pred)
best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
best_val_loss = float(np.min(history.history["val_loss"]))

print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_accuracy:.4f}")
print("\nPrecision / Recall / F1-score:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):")
print(matrix)

## 7. Lưu kết quả để so sánh

Lưu checkpoint Keras, metrics JSON, lịch sử CSV, confusion matrix dạng CSV và hai biểu đồ PNG (đường cong huấn luyện, ma trận nhầm lẫn) vào thư mục outputs của Model 2.

In [ ]:
results = {
    "model": f"Model 2 - {MODEL_NAME.replace('_', ' ').title()}",
    "dataset": DATASET,
    "params": int(model.count_params()),
    "img_size": IMAGE_SIZE[0],
    "epochs_run": len(history.history["loss"]),
    "train_time_min": round(train_time / 60, 2),
    "best_val_loss": best_val_loss,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_macro_precision": float(report["macro avg"]["precision"]),
    "test_macro_recall": float(report["macro avg"]["recall"]),
    "test_macro_f1": float(report["macro avg"]["f1-score"]),
    "test_weighted_f1": float(report["weighted avg"]["f1-score"]),
    "classes": class_names,
    "classification_report": report,
}
with (OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

history_rows = zip(
    history.history["loss"],
    history.history["val_loss"],
    history.history["accuracy"],
    history.history["val_accuracy"],
    history.history.get("learning_rate", [""] * len(history.history["loss"])),
)
with (OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_history.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["train_loss", "val_loss", "train_acc", "val_acc", "lr"])
    writer.writerows(history_rows)

with (OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_confusion_matrix.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["true/predicted", *class_names])
    writer.writerows([[name, *row] for name, row in zip(class_names, matrix)])

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history.history["loss"]) + 1)
axes[0].plot(epochs, history.history["loss"], label="train")
axes[0].plot(epochs, history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(epochs, history.history["accuracy"], label="train")
axes[1].plot(epochs, history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_curves.png", dpi=150)
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(6, 5))
image = axis.imshow(matrix, cmap="Blues")
axis.set(
    xticks=range(len(class_names)),
    yticks=range(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="Predicted label",
    ylabel="True label",
    title=f"{MODEL_NAME} - {DATASET} confusion matrix",
)
figure.colorbar(image, ax=axis)
threshold = matrix.max() / 2 if matrix.size else 0
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(
            column,
            row,
            str(matrix[row, column]),
            ha="center",
            va="center",
            color="white" if matrix[row, column] > threshold else "black",
        )
figure.tight_layout()
figure.savefig(OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_confusion_matrix.png", dpi=150)
plt.show()
plt.close(figure)

print(f"Checkpoint: {model_path}")
print(f"Results saved in: {OUTPUT_DIR}")